In [ ]:
from pathlib import Path

DATA = Path("../data")
for f in sorted(DATA.rglob("*.sql.gz")):
    print(f, f"{f.stat().st_size / 1e6:.1f} MB")

In [ ]:
import gzip

lt_path = next(DATA.rglob("*linktarget.sql.gz"))

with gzip.open(lt_path, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("INSERT INTO"):
            break
        if line.strip() and not line.startswith(("--", "/*")):
            print(line, end="")

In [ ]:
from collections import Counter
from itertools import islice

from wikigraph.ingest.sql_dump import iter_rows

for row in islice(iter_rows(lt_path), 5):
    print(row)

ns_count = Counter()
for _, ns, _ in iter_rows(lt_path):
    ns_count[int(ns)] += 1

print("total rows: ", sum(ns_count.values()))
print(ns_count.most_common(10))

In [ ]:
def show_schema(path):
    """Print the CREATE TABLE statement of a dump file"""
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("INSERT INTO"):
                break
            if line.strip() and not line.startswith(("--", "/*")):
                print(line, end="")


RAW = DATA / "raw" / "simplewiki" / "latest"
show_schema(RAW / "redirect.sql.gz")
print("\n" + "=" * 60 + "\n")
show_schema(RAW / "pagelinks.sql.gz")

In [ ]:
rd_path = RAW / "redirect.sql.gz"

for row in islice(iter_rows(rd_path), 5):
    print(row)

rd_ns = Counter()
interwiki_values = Counter()
fragment_values = Counter()
for _, ns, _, iw, frag in iter_rows(rd_path):
    rd_ns[int(ns)] += 1
    interwiki_values[iw] += 1
    fragment_values[frag == "" or frag == "NULL"] += 1

print("total redirects:", sum(rd_ns.values()))
print("target namespaces:", rd_ns.most_common(5))
print("interwiki values:", interwiki_values.most_common(5))
print("fragment empty? ", fragment_values)

In [ ]:
from wikigraph.ingest.sql_dump import iter_pages

# dict of: title -> page_id, and set of redirects
title_to_id = {}
redirect_ids = set()
for page_id, ns, title, is_redirect in iter_pages(RAW / "page.sql.gz"):
    if ns == 0:
        title_to_id[title] = page_id
        if is_redirect:
            redirect_ids.add(page_id)

# dict of: redirect_from_page_id -> title, where both are in namespace 0 (article)
redirect_target = {}
for rd_from, ns, title, iw, _ in iter_rows(rd_path):
    rd_from = int(rd_from)
    if rd_from in redirect_ids and int(ns) == 0 and iw == "":
        redirect_target[rd_from] = title

print("ns-0 redirect pages:        ", len(redirect_ids))
print("  with usable ns-0 target:  ", len(redirect_target))

missing = 0
double = 0
for _, title in redirect_target.items():
    target = title_to_id.get(title)
    if target is None:
        missing += 1
    elif target in redirect_ids:
        double += 1

print("  target page doesn't exist:", missing)
print("  double redirects:         ", double)

In [ ]:
# dict of: linktarget id → title, articles only
lt_to_title = {}
for lt_id, ns, title in iter_rows(lt_path):
    if ns == "0":
        lt_to_title[int(lt_id)] = title
print("ns-0 link targets:", len(lt_to_title))

pl_path = RAW / "pagelinks.sql.gz"
stats = Counter()
used_targets = set()

for pl_from, from_ns, target_id in iter_rows(pl_path):
    stats["rows"] += 1
    if from_ns != "0":
        stats["source not an article"] += 1
        continue
    pl_from = int(pl_from)
    if pl_from in redirect_ids:
        stats["source is a redirect"] += 1
        continue
    target_id = int(target_id)
    title = lt_to_title.get(target_id)
    if title is None:
        stats["target not in ns"] += 1
        continue
    used_targets.add(target_id)
    target = title_to_id.get(title)
    if target is None:
        stats["red link"] += 1
        continue
    if target in redirect_ids:
        rd_title = redirect_target.get(target)
        if rd_title is None:
            stats["unusable redirect"] += 1
            continue
        target = title_to_id[rd_title]
        stats["via redirect"] += 1
    if target == pl_from:
        stats["self loop"] += 1
        continue
    stats["EDGE"] += 1

for key, value in stats.items():
    print(f"{key:24} {value:>12,}")

unused = len(lt_to_title) - len(used_targets)
print(f"\nns-0 targets never used by an article link: {unused:,}")

In [ ]:
red_counts = Counter()
for pl_from, from_ns, target_id in iter_rows(pl_path):
    if from_ns != "0" or int(pl_from) in redirect_ids:
        continue
    title = lt_to_title.get(int(target_id))
    if title is not None and title not in title_to_id:
        red_counts[title] += 1

print("distinct red-link titles:", len(red_counts))
for title, n in red_counts.most_common(20):
    print(f"{n:>8,}  {title}")

In [ ]:
from wikigraph.ingest.sql_dump import iter_linktargets, iter_pagelinks, iter_redirects

print(next(iter_linktargets(lt_path)))
print(next(iter_redirects(rd_path)))
print(next(iter_pagelinks(pl_path)))

print(sum(1 for _ in iter_linktargets(lt_path)), "expected 3287865")
print(sum(1 for _ in iter_redirects(rd_path)), "expected 134502")
print(sum(1 for _ in iter_pagelinks(pl_path)), "expected 25470883")